# SYNTHESIS-ZERO: Materials Project model experiment

This notebook evaluates the saved model against a small live Materials Project batch. It never prints or stores `MP_API_KEY`. Set a replacement key in the PowerShell process that launches Jupyter:

```powershell
$env:MP_API_KEY = "your replacement key"
```

The notebook performs inference only and does not update production weights.

In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("..")
PROJECT_ROOT = PROJECT_ROOT.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "Data"
CHECKPOINT_PATH = DATA_DIR / "mcrsa_checkpoint.pt"
DATASET_PATH = DATA_DIR / "perovskites_3d_dataset.json"
print(f"Project root: {PROJECT_ROOT}")
print(f"Checkpoint exists: {CHECKPOINT_PATH.exists()}")
print(f"Dataset exists: {DATASET_PATH.exists()}")

Project root: C:\Users\shakti singh\OneDrive\Desktop\Synthesis zero
Checkpoint exists: True
Dataset exists: True


In [7]:
from src.config import load_local_env
import importlib
import src.data_ingestion.live_materials_project as live_materials_project
import src.data_ingestion.materials_project_pipeline as materials_project_pipeline
importlib.reload(live_materials_project)
importlib.reload(materials_project_pipeline)
MaterialsProjectClient = live_materials_project.MaterialsProjectClient
MaterialsProjectPipeline = materials_project_pipeline.MaterialsProjectPipeline
from src.features import MaterialFeaturePipeline
from src.models import MCRSA_EGNN

load_local_env(PROJECT_ROOT / ".env")
api_key = os.environ.get("MP_API_KEY", "").strip()
if not api_key:
    raise RuntimeError("MP_API_KEY is empty. Paste a replacement key into the project .env file, then restart the notebook kernel.")
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Saved model checkpoint not found: {CHECKPOINT_PATH}")

print("MP_API_KEY detected without displaying its value.")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
feature_pipeline = MaterialFeaturePipeline()
model = MCRSA_EGNN(
    in_features=feature_pipeline.node_feature_dim,
    global_features=feature_pipeline.global_feature_dim,
).to(device)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded {CHECKPOINT_PATH.name} on {device}; steps={checkpoint.get('total_steps', 'unknown')}")

MP_API_KEY detected without displaying its value.
Loaded mcrsa_checkpoint.pt on cpu; steps=4611


In [8]:
with DATASET_PATH.open("r", encoding="utf-8") as handle:
    local_dataset = json.load(handle)
train_targets = torch.tensor([float(item["target_energy"]) for item in local_dataset], dtype=torch.float64)
target_mean = float(train_targets.mean())
target_std = max(float(train_targets.std(unbiased=False)), 1e-6)
print(f"Normalization: mean={target_mean:.6f}, std={target_std:.6f} eV/atom")

Normalization: mean=1.470932, std=0.742482 eV/atom


In [11]:
import importlib
import src.data_ingestion.live_materials_project as live_materials_project
importlib.reload(live_materials_project)
MaterialsProjectClient = live_materials_project.MaterialsProjectClient
supported_elements = {"Ba", "Sr", "Ca", "La", "K", "Cs", "Na", "Ti", "Zr", "Hf", "Fe", "Mn", "Nb", "Ta", "Al", "O", "F", "N", "S"}
client = MaterialsProjectClient(api_key, supported_elements=supported_elements)
raw_records = client.fetch_batch(skip=0, limit=5)
print(f"Fetched {len(raw_records)} records from Materials Project.")
if not raw_records:
    raise RuntimeError("No records returned. The client was reloaded; check the API response or key configuration.")
raw_records[:2]

Fetched 5 records from Materials Project.


[{'formation_energy_per_atom': -0.9933755762499998,
  'elements': ['N', 'Na', 'O'],
  'formula_pretty': 'NaNO3',
  'material_id': 'mp-aaahikll',
  'structure': {'@module': 'pymatgen.core.structure',
   '@class': 'Structure',
   'charge': 0.0,
   'lattice': {'matrix': [[5.058565466622774,
      0.0105360098797626,
      0.0056935830082176],
     [2.518834696280307, 5.605701922711505, -0.0507719950626909],
     [2.510195182105497, 1.1889015570196164, 16.915998687375637]],
    'pbc': [True, True, True],
    'a': 5.058579642992699,
    'b': 6.145811587492706,
    'c': 17.142507936583947,
    'alpha': 83.38939263361435,
    'beta': 81.5071648512648,
    'gamma': 65.68612530621164,
    'volume': 479.4757792710996},
   'properties': {},
   'sites': [{'species': [{'element': 'Na', 'occu': 1.0}],
     'abc': [0.0005089412488246, 0.0031974754416595, 0.9987797118265931],
     'properties': {'magmom': 0.0},
     'label': 'Na',
     'xyz': [2.5177604453206306, 1.2053802109513996, 16.895196849727977

In [14]:
import importlib
import src.data_ingestion.materials_project_pipeline as materials_project_pipeline
importlib.reload(materials_project_pipeline)
MaterialsProjectPipeline = materials_project_pipeline.MaterialsProjectPipeline
pipeline = MaterialsProjectPipeline(
    str(DATA_DIR / "notebook_ingestion_audit.jsonl"),
    str(DATA_DIR / "notebook_accepted_records.json"),
    supported_elements,
)
accepted_records = pipeline.normalize_for_inference(raw_records)
print({"fetched": len(raw_records), "validated_for_inference": len(accepted_records)})
if not accepted_records:
    raise RuntimeError("The API batch contained no records valid for inference.")

AttributeError: 'MaterialsProjectPipeline' object has no attribute 'normalize_for_inference'

In [ ]:
rows = []
with torch.no_grad():
    for record in accepted_records:
        elements = record["elements"]
        coordinates = record["coordinates_3d"]
        node_features, global_features = feature_pipeline.tensors(elements, coordinates, device)
        coords = torch.tensor(coordinates, dtype=torch.float32, device=device)
        normalized_prediction, _, uncertainty = model(node_features, coords, global_features)
        prediction = float(normalized_prediction.item() * target_std + target_mean)
        uncertainty_eV = float(uncertainty.item() * target_std)
        target = float(record["target_energy"])
        rows.append({
            "material_id": record.get("material_id"),
            "formula": record["formula"],
            "api_target_energy_eV_atom": target,
            "predicted_energy_eV_atom": prediction,
            "absolute_error_eV_atom": abs(prediction - target),
            "uncertainty_eV_atom": uncertainty_eV,
            "status": "inference_complete",
        })

results = pd.DataFrame(rows).sort_values("absolute_error_eV_atom")
display(results)
print(f"Mean absolute error: {results['absolute_error_eV_atom'].mean():.6f} eV/atom")

## Interpretation

- `predicted_energy_eV_atom` is the model output, not a new API label.
- This small sample is an experiment, not a production benchmark, and may overlap with training data.
- `uncertainty_eV_atom` is the learned uncertainty head scaled to eV/atom; treat it as a relative signal until calibrated.
- The notebook performs inference only and does not update the checkpoint.